# Install the required libraries

In [ ]:
!pip install -q -U transformers accelerate
!pip install -q peft trl bitsandbytes datasets scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 96.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 721.6/721.6 kB 19.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 19.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 36.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 17.9 MB/s eta 0:00:00


#Import the Python libraries

In [ ]:
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model
from trl import SFTTrainer, SFTConfig
from transformers import TrainingArguments
from sklearn.metrics import accuracy_score, f1_score
from tqdm.notebook import tqdm

# Load Qwen3.5-0.8B with 4-bit quantization

In [ ]:
model_id = "Qwen/Qwen3.5-0.8B"

bnb_config = BitsAndBytesConfig(
  load_in_4bit=True,
  bnb_4bit_quant_type="nf4",
  bnb_4bit_compute_dtype=torch.bfloat16
)

model = AutoModelForCausalLM.from_pretrained(
  model_id,
  quantization_config=bnb_config,
  device_map="auto",
  dtype=torch.bfloat16
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

model.safetensors-00001-of-00001.safeten(…):   0%|          | 0.00/1.75G [00:00<?, ?B/s]

[transformers] The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

# Load the tokenizer

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token

model.config.pad_token_id = tokenizer.eos_token_id
model.generation_config.pad_token_id = tokenizer.eos_token_id

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/12.8M [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

# Load the wolof dataset

In [ ]:
dataset = load_dataset("vonewman/alpaca-sharegpt-wolof", split="train")

README.md:   0%|          | 0.00/388 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/47463 [00:00<?, ? examples/s]

In [ ]:
dataset[0]

{'conversation': [{'from': 'human',
   'value': 'Jox ñatti xeeti feem ngir am wérgi-yaram.'},
  {'from': 'gpt',
   'value': '1. Lekk regime bu dëgër te bari ferñent: Fexeel nga am ay fruit aki lujum yu bari, protein yu amul yaram, pepp yu mat sëkk, ak grees yu baax. Loolu dafay jàppale sa yaram mu am ferñent yi war ngir mëna doxal bu baax, te mën nala jàppale nga moytu feebar bu kronik.\\n\\n 2. Deel faral di tàggat sa yaram: Tàggat yaram lu am solo la ngir am yax yu dëgër, sidit yi ak wérgi-yaramu xol ak deret. Fexeel def 150 simili ci tàggat yaram bu yemmamaay wala 75 simili ci tàggat yaram bu am doole ayu-bis bu nekk.\\n\\n 3. Nelaw lu doy: Nelaw lu doy lu am solo la ci wérgi-yaramu yaram ak xel. Dafay baaxal sa xol, gëna suqali xel mi, ba noppi jàppale màgg gu sell gi ak doxalinu immuniteer bi. Fexeel nelaw 7 ba 9 waxtu guddi gu nekk.'}],
 'source': 'alpaca'}

In [ ]:
dataset = dataset.remove_columns(['source'])


# Split train and test data

In [ ]:
train_dataset = dataset.select(range(1000))
test_dataset = dataset.select(range(1000, 1200))

In [ ]:
print(train_dataset[0])
print(test_dataset[0])

{'conversation': [{'from': 'human', 'value': 'Jox ñatti xeeti feem ngir am wérgi-yaram.'}, {'from': 'gpt', 'value': '1. Lekk regime bu dëgër te bari ferñent: Fexeel nga am ay fruit aki lujum yu bari, protein yu amul yaram, pepp yu mat sëkk, ak grees yu baax. Loolu dafay jàppale sa yaram mu am ferñent yi war ngir mëna doxal bu baax, te mën nala jàppale nga moytu feebar bu kronik.\\n\\n 2. Deel faral di tàggat sa yaram: Tàggat yaram lu am solo la ngir am yax yu dëgër, sidit yi ak wérgi-yaramu xol ak deret. Fexeel def 150 simili ci tàggat yaram bu yemmamaay wala 75 simili ci tàggat yaram bu am doole ayu-bis bu nekk.\\n\\n 3. Nelaw lu doy: Nelaw lu doy lu am solo la ci wérgi-yaramu yaram ak xel. Dafay baaxal sa xol, gëna suqali xel mi, ba noppi jàppale màgg gu sell gi ak doxalinu immuniteer bi. Fexeel nelaw 7 ba 9 waxtu guddi gu nekk.'}]}
{'conversation': [{'from': 'human', 'value': 'Xaajaleel frase bu nekk muy fësal, laaj, santaane, wala yuuxu.\nLu tax ngay def loolu?'}, {'from': 'gpt', '

# Formating Data for qwen

In [ ]:
def format_conversation(examples):
    texts = []
    for text in examples["text"]:
        texts.append(text)
    return {"text": texts}

In [ ]:
train_dataset = train_dataset.map(format_conversation, batched=True)
test_dataset = test_dataset.map(format_conversation, batched=True)

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

In [ ]:
train_dataset[0]

{'conversation': [{'from': 'human',
   'value': 'Jox ñatti xeeti feem ngir am wérgi-yaram.'},
  {'from': 'gpt',
   'value': '1. Lekk regime bu dëgër te bari ferñent: Fexeel nga am ay fruit aki lujum yu bari, protein yu amul yaram, pepp yu mat sëkk, ak grees yu baax. Loolu dafay jàppale sa yaram mu am ferñent yi war ngir mëna doxal bu baax, te mën nala jàppale nga moytu feebar bu kronik.\\n\\n 2. Deel faral di tàggat sa yaram: Tàggat yaram lu am solo la ngir am yax yu dëgër, sidit yi ak wérgi-yaramu xol ak deret. Fexeel def 150 simili ci tàggat yaram bu yemmamaay wala 75 simili ci tàggat yaram bu am doole ayu-bis bu nekk.\\n\\n 3. Nelaw lu doy: Nelaw lu doy lu am solo la ci wérgi-yaramu yaram ak xel. Dafay baaxal sa xol, gëna suqali xel mi, ba noppi jàppale màgg gu sell gi ak doxalinu immuniteer bi. Fexeel nelaw 7 ba 9 waxtu guddi gu nekk.'}],
 'text': '<|im_start|>user\nJox ñatti xeeti feem ngir am wérgi-yaram.<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\n1. Lekk regime bu 

# Configuring LoRA Adapters

In [ ]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=16,
    lora_dropout=0,
    bias='none',
    task_type='CAUSAL_LM',
    target_modules=[
        'q_proj','k_proj','v_proj','o_proj',
        'gate_proj','up_proj','down_proj'
    ]
)

# Define the training configuration

In [ ]:
training_args = SFTConfig(
    output_dir="./qwen35-lora",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    num_train_epochs=2)

# Train the model

In [ ]:
trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    peft_config=lora_config,
    args=training_args,
    processing_class=tokenizer
)
trainer.train()

Adding EOS to train dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 248046}.


Step,Training Loss
1,4.578819
2,4.206368
3,4.065371
4,3.732660
5,4.136711
6,3.407385
7,3.849849
8,3.564381
9,3.907125
10,3.695016


TrainOutput(global_step=250, training_loss=2.759165831565857, metrics={'train_runtime': 2960.8295, 'train_samples_per_second': 0.675, 'train_steps_per_second': 0.084, 'total_flos': 2530551245440512.0, 'train_loss': 2.759165831565857})

# Save the trained adapter

In [ ]:
trainer.model.save_pretrained("qwen35-small-news-class")
tokenizer.save_pretrained("qwen35-small-news-class")

('qwen35-small-news-class/tokenizer_config.json',
 'qwen35-small-news-class/chat_template.jinja',
 'qwen35-small-news-class/tokenizer.json')

# Inference Model

In [ ]:

trainer.model.eval()

prompt = "Jox ñatti xeeti feem ngir am wérgi-yaram."

messages = [{"role": "user", "content": prompt}]

input_text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(input_text, return_tensors="pt").to("cuda")

with torch.inference_mode():
    outputs = trainer.model.generate(
        **inputs,
        max_new_tokens=200,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )

response = tokenizer.decode(
    outputs[0][inputs['input_ids'].shape[1]:],
    skip_special_tokens=True
)
print(response)

1. Xalaat: Xalaat bi dafay jàppale wérgi-yaram ci 1000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
save_path = "/content/drive/MyDrive/qwen35-wolof"

trainer.model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)

print(f"Modèle sauvegardé dans : {save_path}")

Modèle sauvegardé dans : /content/drive/MyDrive/qwen35-wolof


# Evaluate the fine-tuned model

In [ ]:
from transformers import AutoModelForCausalLM
base_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    dtype=torch.bfloat16
)


from peft import PeftModel

model = PeftModel.from_pretrained(
    base_model,
    "/kaggle/working/qwen35-small-news-class"
)

model.eval()
model.print_trainable_parameters()